# TCGA-BRCA: K-means, FCM, and WFCM
One self-contained workflow: load -> match patients -> filter genes -> PCA ->
fit all methods -> evaluate once. No local helper-module imports are required.
WFCM engine "stable" corrects numerical defects in the supplied ADNI code.
Engine "legacy" reproduces the earlier adaptation for auditing only.
These are numerical revisions of the notebook density, not a verified
implementation of an unseen manuscript.

In [ ]:
from pathlib import Path
import json
import hashlib
import platform
import warnings
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib.pyplot as plt
from scipy.optimize import minimize, linear_sum_assignment
from scipy.special import logsumexp, xlogy
from scipy.spatial.distance import cdist, pdist
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from IPython.display import display

SUBTYPES = ['Basal', 'Her2', 'LumA', 'LumB']



## 1. Settings
Set the expression path to your already downloaded matrix. Do not log-transform
it again: the earlier analysis used the loaded Xena values directly.
INPUT_MODE="existing" preserves X_pca and labels already in this kernel exactly.
INPUT_MODE="prepared" loads the prior saved PC CSV.
INPUT_MODE="raw" executes the complete original data preparation below.
Preprocessing confirmed: PCA directly on selected genes; no StandardScaler.

In [ ]:
INPUT_MODE = 'raw'                     # raw / existing / prepared
EXPR_PATH = Path('TCGA-BRCA.star_tpm.tsv.gz')
CLINICAL_SOURCE = ('https://tcga.xenahubs.net/download/'
                   'TCGA.BRCA.sampleMap/BRCA_clinicalMatrix')
PREPARED_PATH = Path('TCGA_BRCA_WFCM_analysis_data.csv')
STANDARDIZE_GENES = False              # Confirmed: PCA directly on the selected genes.
N_HVG, N_PC, K = 5000, 5, 4
EXPECTED_N = 819
EXPECTED_COUNTS = {'Basal': 139, 'Her2': 67, 'LumA': 421, 'LumB': 192}
STRICT_COHORT = True                   # Prevent accidental changes in cohort.
MODE = 'fixed'                        # fixed / select; decide before evaluation.
FIXED_M = 2                            # only used if MODE='fixed'
M_GRID = np.geomspace(1.5, 2.5, 8)        # Original ADNI grid if MODE='select'.
WFCM_ENGINE = 'stable'                 # stable / legacy
WXB_MIN = None                        # No cutoff; 0.1 reproduces ADNI selection.
FCM_SEEDS = [321, 42, 7, 199]  
WFCM_SEEDS = [321, 42, 7, 199]             # multiple restarts improves stability
NUM_IS = 5000                          # need adequate IS coverage in 5D
IS_SEED, VALIDATION_SEED = 674, 675
MAX_OUTER, INNER_MAXITER = 30, 1000
OUTPUT_DIR = Path('TCGA_BRCA_comparison_results')



## 2. Shared data preparation
PAM50 defines the four-subtype evaluation cohort; subtype identities are never
used for feature selection, fitting, initialization, or choosing m/seeds.

### Kaiser–Guttman diagnostic
Retain components whose eigenvalues of the **original feature correlation matrix**
exceed 1. This uses all available components, not only the fitted/truncated PCs.
Constant features are excluded. This is a descriptive sensitivity check; the
paper's existing PCA and clustering dimensions remain unchanged. Correlation PCA
here can differ from the unscaled covariance PCA used in the main analysis.
Reference: [EFAtools KGC documentation](https://search.r-project.org/CRAN/refmans/EFAtools/html/KGC.html).

This diagnostic runs in raw input mode; prepared/existing PCs alone are insufficient.


In [ ]:
def kaiser_guttman(features):
    """Correlation-PCA diagnostic on the complete pre-PCA feature matrix.

    Sample-SD scaling (ddof=1) makes eigenvalues comparable to one.
    This diagnostic does not change the features used for model fitting.
    """
    x = np.asarray(features, dtype=float)
    if x.ndim != 2 or min(x.shape) < 2 or not np.isfinite(x).all():
        raise ValueError('Provide a finite samples-by-features matrix with at least two rows/columns.')
    sd = x.std(axis=0, ddof=1)
    keep = sd > 0
    if not keep.any():
        raise ValueError('No nonconstant features.')
    z = (x[:, keep] - x[:, keep].mean(axis=0)) / sd[keep]
    # Singular values avoid constructing a large genes-by-genes matrix.
    eigenvalues = np.linalg.svd(z, compute_uv=False) ** 2 / (len(z) - 1)
    # Include the zero eigenvalues when features outnumber samples.
    eigenvalues = np.pad(eigenvalues, (0, max(0, int(keep.sum()) - len(eigenvalues))))
    table = pd.DataFrame({'PC': np.arange(1, len(eigenvalues) + 1),
                          'correlation_eigenvalue': eigenvalues,
                          'variance_fraction': eigenvalues / eigenvalues.sum(),
                          'retain_KG': eigenvalues > 1.0})
    table['cumulative_variance'] = table.variance_fraction.cumsum()
    print('Kaiser–Guttman PCs (eigenvalue > 1):', int(table.retain_KG.sum()))
    print('Constant features excluded:', int((~keep).sum()))
    return table


In [ ]:
def load_raw_data(expr_path, clinical_source, standardize_genes):
    if standardize_genes is not True and standardize_genes is not False:
        raise ValueError('Set STANDARDIZE_GENES=True or False to match your prior PCA, or use existing/prepared input mode.')
    if not Path(expr_path).is_file():
        raise FileNotFoundError(f'Set EXPR_PATH to your downloaded STAR TPM matrix: {expr_path}')
    expr = pd.read_csv(expr_path, sep='\t', low_memory=False)
    clinical = pd.read_csv(clinical_source, sep='\t', low_memory=False)
    for col in ['sampleID', 'PAM50Call_RNAseq']:
        if col not in clinical:
            raise ValueError(f'Clinical table missing {col}')
    if 'Ensembl_ID' not in expr:
        raise ValueError('Expected expression gene column Ensembl_ID.')
    genes = expr.set_index('Ensembl_ID')
    if not genes.index.is_unique or not genes.columns.is_unique:
        raise ValueError('Duplicate expression identifiers; inspect before proceeding.')
    info = pd.DataFrame({'sample_id': genes.columns})
    if not info.sample_id.str.match(r'^TCGA-[^-]+-[^-]+-\d{2}').all():
        raise ValueError('Unexpected TCGA sample barcode format.')
    info['patient_id'] = info.sample_id.str[:12]
    info['sample_type'] = info.sample_id.str.split('-').str[3].str[:2]
    primary = (info.loc[info.sample_type.eq('01')]
               .sort_values(['patient_id', 'sample_id'])
               .drop_duplicates('patient_id', keep='first'))
    pam = clinical[['sampleID', 'PAM50Call_RNAseq']].rename(
        columns={'sampleID': 'clinical_sample_id', 'PAM50Call_RNAseq': 'PAM50'}).copy()
    # Expression uses vial IDs (...-01A); the clinical table uses sample IDs
    # (...-01). Match at the clinical table's sample level, not patient level.
    primary = primary.copy()
    primary['sample_key'] = primary.sample_id.str[:15]
    pam['sample_key'] = pam.clinical_sample_id.str[:15]
    pam['patient_id'] = pam.clinical_sample_id.str[:12]
    selected = pam.loc[pam.sample_key.isin(primary.sample_key)].copy()
    conflicts = selected.groupby('sample_key').PAM50.nunique(dropna=True)
    report_dir = Path(globals().get('OUTPUT_DIR', 'TCGA_BRCA_comparison_results'))
    report_dir.mkdir(parents=True, exist_ok=True)
    if (conflicts > 1).any():
        bad = selected.loc[selected.sample_key.isin(conflicts[conflicts > 1].index)]
        report = report_dir / 'conflicting_selected_sample_annotations.csv'
        bad.to_csv(report, index=False)
        raise ValueError('Conflicting calls for the SAME selected tumor sample; '
                         f'inspect {report.resolve()}')
    # Consistent repeated records are safe to collapse. Prefer a known call to
    # a missing duplicate, but never substitute an annotation from another sample.
    annotation = (selected.assign(_missing=selected.PAM50.isna())
                  .sort_values('_missing', kind='stable')
                  .drop_duplicates('sample_key', keep='first'))
    info = primary.merge(annotation[['sample_key', 'PAM50']], on='sample_key',
                         how='left', validate='one_to_one')
    legacy = pam.drop_duplicates('patient_id', keep='first').set_index('patient_id').PAM50
    info['previous_patient_first_PAM50'] = info.patient_id.map(legacy)
    info['label_changed_vs_patient_first'] = (
        info.PAM50.fillna('<missing>') != info.previous_patient_first_PAM50.fillna('<missing>'))
    info['annotation_status'] = np.where(info.PAM50.notna(), 'sample_matched',
        np.where(info.sample_key.isin(annotation.sample_key), 'missing_PAM50', 'unmatched_sample'))
    info['included_in_analysis'] = info.PAM50.isin(SUBTYPES)
    report = report_dir / 'sample_annotation_audit.csv'
    info.to_csv(report, index=False)
    print('Sample annotation matching:', info.annotation_status.value_counts().to_dict())
    print('Labels differing from the old patient-first merge:',
          int(info.label_changed_vs_patient_first.sum()))
    print('Matching report:', report.resolve())
    info = info.loc[info.PAM50.isin(SUBTYPES)].set_index('sample_id')
    X_gene = genes.loc[:, info.index].T.copy()
    info = info.loc[X_gene.index]
    y = info.PAM50.copy()
    X_gene = X_gene.replace([np.inf, -np.inf], np.nan)
    X_gene = X_gene.loc[:, X_gene.notna().all(axis=0)]
    var = X_gene.var(axis=0)
    X_gene = X_gene.loc[:, var > 1e-8]
    chosen = X_gene.var(axis=0).sort_values(ascending=False).head(N_HVG).index
    hvg = X_gene.loc[:, chosen]
    if hvg.shape[1] < N_PC:
        raise ValueError('Too few valid genes for requested PCA.')
    # No extra prevalence filter, log transform, PC scaling, or whitening.
    scaler = StandardScaler() if standardize_genes else None
    features = scaler.fit_transform(hvg) if scaler is not None else hvg.to_numpy()
    global kg_tcga
    kg_tcga = kaiser_guttman(hvg.to_numpy())
    kg_tcga.to_csv(report_dir / 'kaiser_guttman.csv', index=False)
    display(kg_tcga.head(20))
    pca = PCA(n_components=N_PC, random_state=123)
    pcs = pca.fit_transform(features)
    return pcs, y, info, pca, list(chosen)


def check_input(X, y):
    if not isinstance(y, pd.Series):
        raise ValueError('Provide labels as a Series indexed by patient/sample IDs to preserve alignment.')
    if isinstance(X, pd.DataFrame) and not X.index.equals(y.index):
        raise ValueError('X_pca and labels indices differ; align explicitly first.')
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or X.shape != (len(y), N_PC) or not np.isfinite(X).all():
        raise ValueError(f'Expected finite patients x {N_PC} PC matrix matching labels.')
    if not y.index.is_unique or y.isna().any() or set(y) != set(SUBTYPES):
        raise ValueError('Invalid sample IDs or PAM50 labels.')
    if STRICT_COHORT and (len(y) != EXPECTED_N or y.value_counts().to_dict() != EXPECTED_COUNTS):
        raise ValueError('Cohort changed. Compare IDs and preprocessing before fitting: ' + str(y.value_counts().to_dict()))
    return X.copy(), y.copy()



## 3. Original ADNI implementation (audit option)
These functions are embedded to avoid import-path errors. The stable engine
below is the default; legacy functions retain the known epsilon defects.

In [ ]:
"""ADNI-notebook WFCM adapted to an already-preprocessed PAM50 PC matrix.

Dependencies: numpy, pandas, scipy, scikit-learn, matplotlib (plots only).
Default numerical equations, optimization, seeds, and XB cutoff reproduce the
notebook methodology. See WFCM_methodology.md for inherited limitations.
U is patients x clusters; U_wfcm = result['U'].T matches skfuzzy orientation.
"""
import warnings
import numpy as np
import pandas as pd
from scipy.optimize import minimize, linear_sum_assignment
from scipy.spatial.distance import cdist, pdist
from scipy.special import xlogy
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

SUBTYPES = ['Basal', 'Her2', 'LumA', 'LumB']


def _powers(X, centers, w, m, epsilon=1e-8):
    # Deliberately retain ADNI's (Euclidean distance + epsilon)^2.
    return (w * (cdist(X, centers) + epsilon)**2)**(-1.0 / (m - 1))


def membership(X, centers, w, m):
    a = _powers(X, centers, w, m)
    # The notebook adds epsilon here, so row sums may be slightly below one.
    return a / (a.sum(axis=1, keepdims=True) + 1e-8)


def _unpack(params, K, p):
    return params[0], params[1:1+K*p].reshape(K, p), params[1+K*p:]


def negative_log_likelihood(params, X, proposal_samples, proposal_pdf, m, K):
    """Actual cell-5 likelihood; U is absent because the original ignores U."""
    eps = 1e-8
    sigma, centers, w = _unpack(params, K, X.shape[1])
    with np.errstate(over='ignore', divide='ignore', invalid='ignore'):
        s_data = _powers(X, centers, w, m).sum(axis=1) + eps
        energy = np.sum(1.0 / (sigma**2 * (s_data**(m-1) + eps)))
        s_prop = _powers(proposal_samples, centers, w, m).sum(axis=1) + eps
        f = np.exp(-1.0 / (sigma**2 * s_prop**(m-1) + eps))
        z = np.mean(f / (proposal_pdf + eps)) + eps
        value = len(X) * np.log(z) + energy
    return float(value)


def weighted_xie_beni(X, U, centers, m, w):
    """Cell 5: sum_ij K*w_j*U_ij^m*d_ij^2 / (N*min center distance^2)."""
    sep = pdist(centers, metric='sqeuclidean').min()
    if not np.isfinite(sep) or sep < 1e-14:
        return np.inf
    return float(np.sum(len(w) * w * U**m * cdist(X, centers, 'sqeuclidean'))
                 / (len(X) * sep))


def fit_wfcm(X, proposal_samples, proposal_pdf, m, K=4,
             random_state=321, max_iter=20, tol=1e-5):
    """Preserve ADNI's centroid step, full L-BFGS-B step, then normalize w.

    A failed optimizer leaves the centroid-step result in place, as in ADNI,
    but is reported. Final U and NLL are recomputed (fixes stale returns).
    """
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or not np.isfinite(X).all() or len(X) < K or K < 2:
        raise ValueError('X must be a finite N x p matrix with N >= K >= 2.')
    if not np.isfinite(m) or m <= 1 or max_iter < 1:
        raise ValueError('Require m > 1 and max_iter >= 1.')
    rng = np.random.RandomState(random_state)
    centers = X[rng.choice(len(X), K, replace=False)].copy()
    sigma, w, old_nll = 1.0, np.ones(K)/K, np.inf
    bounds = [(1e-5, 100.)] + [(None, None)]*(K*X.shape[1]) + [(0., 1.)]*K
    history = []
    converged = False
    for iteration in range(1, max_iter+1):
        U = membership(X, centers, w, m)
        um = U**m
        centers = (um.T @ X) / (um.sum(axis=0)[:, None] + 1e-8)
        theta = np.r_[sigma, centers.ravel(), w]
        opt = minimize(negative_log_likelihood, theta,
                       args=(X, proposal_samples, proposal_pdf, m, K),
                       method='L-BFGS-B', bounds=bounds)
        if opt.success:
            sigma, centers, w = _unpack(opt.x, K, X.shape[1])
            w = w / (w.sum() + 1e-12)
        nll = negative_log_likelihood(np.r_[sigma, centers.ravel(), w], X,
                                      proposal_samples, proposal_pdf, m, K)
        history.append(dict(iteration=iteration, NLL=nll,
                            optimizer_success=bool(opt.success),
                            optimizer_message=str(opt.message)))
        converged = bool(np.isfinite(nll) and abs(old_nll-nll) < tol)
        old_nll = nll
        if converged or not np.isfinite(nll):
            break
    U_raw = membership(X, centers, w, m)
    sums = U_raw.sum(axis=1, keepdims=True)
    valid = (np.isfinite(U_raw).all() and np.isfinite(nll)
             and np.all(sums > 0) and np.all(w > 0))
    # Use raw memberships for ADNI WXB; normalize for probability summaries.
    U = U_raw / sums if valid else np.full_like(U_raw, np.nan)
    return dict(sigma=sigma, centers=centers, w=w, m=float(m), U=U,
                U_raw=U_raw, NLL=nll, converged=converged, valid=bool(valid),
                n_iter=iteration, history=pd.DataFrame(history),
                optimizer_failures=sum(not h['optimizer_success'] for h in history),
                max_membership_row_deficit=float(np.max(np.abs(sums-1))))


def select_wfcm_m(X_pca, K=4, m_grid=None, num_proposals=None,
                  proposal_seed=674, fit_seed=321, max_iter=20,
                  xb_min=0.1, verbose=True):
    """Unsupervised selection: this function never accepts PAM50 labels.

    Defaults from ADNI cell 7: geomspace(1.1,6,8), N proposal draws from
    a 2-component full-covariance GMM, one fit per m, exclude WXB < 0.1.
    Set xb_min=None only for an explicitly changed, pure-WXB-minimum protocol.
    """
    X = np.asarray(X_pca, dtype=float)
    if X.ndim != 2 or not np.isfinite(X).all() or len(X) < K or K < 2:
        raise ValueError('Expected finite N x p input, N >= K >= 2.')
    m_grid = np.geomspace(1.1, 6, 8) if m_grid is None else np.asarray(m_grid)
    if np.ndim(m_grid) != 1 or not len(m_grid) or np.any(~np.isfinite(m_grid)) or np.any(m_grid <= 1):
        raise ValueError('m_grid must contain finite values > 1.')
    n_prop = len(X) if num_proposals is None else int(num_proposals)
    gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=proposal_seed)
    gmm.fit(X)
    samples, _ = gmm.sample(n_prop)
    logq = gmm.score_samples(samples)
    q = np.exp(logq)
    floor_fraction = float(np.mean(q < 1e-8))
    if floor_fraction > 0:
        warnings.warn(f'{floor_fraction:.1%} of proposal densities are below ADNI epsilon=1e-8. '
                      'The inherited q+epsilon likelihood may be distorted; inspect methodology notes.')
    fits, rows = [], []
    for m in m_grid:
        fit = fit_wfcm(X, samples, q, float(m), K, fit_seed, max_iter)
        xb = weighted_xie_beni(X, fit['U_raw'], fit['centers'], m, fit['w']) if fit['valid'] else np.inf
        eligible = bool(np.isfinite(xb) and (xb_min is None or xb >= xb_min))
        fit['WXB'] = xb
        fits.append(fit)
        rows.append(dict(m=float(m), WXB=xb, eligible=eligible, NLL=fit['NLL'],
                         converged=fit['converged'], optimizer_failures=fit['optimizer_failures'],
                         n_iter=fit['n_iter'], min_weight=float(fit['w'].min()),
                         n_hard_clusters=len(np.unique(fit['U'].argmax(axis=1))) if fit['valid'] else 0,
                         max_membership_row_deficit=fit['max_membership_row_deficit']))
        if verbose:
            print(f"m={m:.5f}, WXB={xb:.6g}, eligible={eligible}, converged={fit['converged']}", flush=True)
    table = pd.DataFrame(rows)
    eligible_indices = table.index[table.eligible]
    if not len(eligible_indices):
        raise RuntimeError('No eligible fit under the ADNI selection rule. Diagnostics:\n' + table.to_string(index=False))
    best_index = table.loc[eligible_indices, 'WXB'].idxmin()
    best = fits[best_index]
    if not best['converged'] or best['optimizer_failures']:
        warnings.warn('Selected fit has convergence/optimizer issues; inspect its history before interpreting.')
    best['proposal_diagnostics'] = dict(num_samples=n_prop, pdf_below_epsilon_fraction=floor_fraction,
                                        min_log_pdf=float(logq.min()), max_log_pdf=float(logq.max()),
                                        gmm_converged=bool(gmm.converged_))
    return best, table, fits




## 4. Stable WFCM and ordinary FCM
WFCM retains the density exp[-h(x)/sigma^2], with
h(x)=[sum_j (w_j*||x-v_j||^2)^(-1/(m-1))]^(1-m).
log-sum-exp computes h and memberships without absolute additive floors.
Weights sum to one THROUGHOUT optimization via K-1 anchored log-ratios.
Numerical log-ratio bounds [-30,30] prevent floating-point weight collapse.
Centers, sigma, and weights remain likelihood-estimated (not size-based weights).

In [ ]:
def stable_terms(X, centers, logw, m):
    if m <= 1:
        raise ValueError('m must be strictly greater than 1.')
    d2 = cdist(X, centers, 'sqeuclidean')
    safe = np.maximum(d2, 1e-200)   # Only exact/numerically zero distances.
    scores = -(np.log(safe) + logw) / (m-1)
    log_s = logsumexp(scores, axis=1)
    U = np.exp(scores-log_s[:, None])
    h = np.exp(-(m-1)*log_s)
    return U, h, safe


def unpack_stable(theta, K, p):
    log_sigma = theta[0]
    centers = theta[1:1+K*p].reshape(K, p)
    logits = np.r_[theta[1+K*p:], 0.0]
    logw = logits-logsumexp(logits)
    return log_sigma, centers, logw


def nll_gradient(theta, X, Z, logq, m, K):
    log_sigma, centers, logw = unpack_stable(theta, K, X.shape[1])
    w = np.exp(logw)
    def terms(A):
        U, h, d2 = stable_terms(A, centers, logw, m)
        return U, h*np.exp(-2*log_sigma), d2
    ux, ex, dx = terms(X)
    uz, ez, dz = terms(Z)
    logratio = -ez-logq
    lse = logsumexp(logratio)
    prob = np.exp(logratio-lse)
    value = len(X)*(lse-np.log(len(Z))) + ex.sum()
    # dNLL = sum(data energy gradients) - N*IS-weighted proposal gradients.
    def sum_grad(A, U, e, d2, a):
        t = a[:, None] * e[:, None] * U / d2
        centers_grad = 2*(centers*t.sum(axis=0)[:, None]-t.T @ A)
        weight_grad = ((a*e)[:, None]*(U-w)).sum(axis=0)[:-1]
        return np.r_[-2*np.dot(a, e), centers_grad.ravel(), weight_grad]
    grad = sum_grad(X, ux, ex, dx, np.ones(len(X))) - sum_grad(Z, uz, ez, dz, len(X)*prob)
    return float(value), grad


def importance_diagnostics(theta, Z, logq, m, K, p):
    logsigma, centers, logw = unpack_stable(theta, K, p)
    _, h, _ = stable_terms(Z, centers, logw, m)
    lr = -h*np.exp(-2*logsigma)-logq
    log_z = logsumexp(lr)-np.log(len(Z))
    normalized = np.exp(lr-logsumexp(lr))
    return {'logZ': float(log_z), 'ESS': float(1/np.sum(normalized**2)),
            'ESS_fraction': float(1/(len(Z)*np.sum(normalized**2))),
            'largest_IS_weight': float(normalized.max()),
            'legacy_pdf_floor_fraction': float(np.mean(logq < np.log(1e-8)))}


def fit_wfcm_stable(X, Z, logq, m, K=4, seed=321, max_outer=20, inner_maxiter=300):
    n, p = X.shape
    centers = X[np.random.RandomState(seed).choice(n, K, replace=False)].copy()
    theta = np.r_[0., centers.ravel(), np.zeros(K-1)]  # sigma=1, equal weights
    bounds = [(np.log(1e-5), np.log(100.))] + [(None, None)]*(K*p) + [(-30., 30.)]*(K-1)
    value, grad = nll_gradient(theta, X, Z, logq, m, K)
    history = []
    converged = False
    for it in range(1, max_outer+1):
        previous = value
        logsigma, centers, logw = unpack_stable(theta, K, p)
        U, _, _ = stable_terms(X, centers, logw, m)
        # U^m centroid proposal, normalized in log space within each cluster.
        logum = m*np.log(np.maximum(U, np.finfo(float).tiny))
        normalized_um = np.exp(logum-logsumexp(logum, axis=0))
        candidate = theta.copy()
        candidate[1:1+K*p] = (normalized_um.T @ X).ravel()
        proposed, _ = nll_gradient(candidate, X, Z, logq, m, K)
        # A centroid update minimizes fuzzy distortion, not this likelihood.
        # Accept only if it improves the SAME fixed-sample NLL.
        centroid_accepted = bool(np.isfinite(proposed) and proposed < value)
        start = candidate if centroid_accepted else theta
        opt = minimize(nll_gradient, start, args=(X, Z, logq, m, K),
                       method='L-BFGS-B', jac=True, bounds=bounds,
                       options={'maxiter': inner_maxiter, 'ftol': 1e-12, 'gtol': 1e-6, 'maxls': 40})
        trial = opt.x
        trial_value, trial_grad = nll_gradient(trial, X, Z, logq, m, K)
        if not np.isfinite(trial_value):
            trial = start
            trial_value, trial_grad = nll_gradient(trial, X, Z, logq, m, K)
        accepted = bool(np.isfinite(trial_value) and trial_value <= value)
        if accepted:
            theta, value, grad = trial.copy(), trial_value, trial_grad
        history.append({'iteration': it, 'NLL': value, 'optimizer_success': bool(opt.success),
                        'optimizer_message': str(opt.message), 'step_accepted': accepted,
                        'centroid_accepted': centroid_accepted,
                        'gradient_inf_norm': float(np.max(np.abs(grad)))})
        converged = bool(accepted and opt.success and abs(previous-value) < 1e-5)
        if converged or not accepted:
            break
    logsigma, centers, logw = unpack_stable(theta, K, p)
    U, _, _ = stable_terms(X, centers, logw, m)
    return dict(U=U, centers=centers, w=np.exp(logw), sigma=float(np.exp(logsigma)),
                m=float(m), NLL=value, theta=theta, converged=converged,
                history=pd.DataFrame(history), seed=seed, engine='stable',
                optimizer_failures=sum(not row['optimizer_success'] for row in history),
                weight_bound_hit=bool(np.any(np.abs(theta[1+K*p:]) > 29.99)),
                sigma_bound_hit=bool(abs(logsigma-np.log(1e-5)) < 1e-5 or abs(logsigma-np.log(100)) < 1e-5))


def fit_fcm(X, m, K=4, seed=123, maxiter=1000, error=1e-6):
    # Standard fuzzy c-means, random membership initialization as in prior code.
    if m <= 1:
        raise ValueError('m must be greater than one.')
    rng = np.random.RandomState(seed)
    U = rng.rand(K, len(X)).T
    U /= U.sum(axis=1, keepdims=True)
    logw = np.full(K, -np.log(K))
    converged = False
    for it in range(1, maxiter+1):
        logum = m*np.log(np.maximum(U, np.finfo(float).tiny))
        weights = np.exp(logum-logsumexp(logum, axis=0))
        centers = weights.T @ X
        updated, _, _ = stable_terms(X, centers, logw, m)
        delta = np.linalg.norm(updated-U)
        U = updated
        if delta < error:
            converged = True
            break
    objective = float(np.sum(U**m * cdist(X, centers, 'sqeuclidean')))
    return dict(U=U, centers=centers, w=np.ones(K)/K, m=float(m), seed=seed,
                objective=objective, converged=converged, n_iter=it)


def select_fcm(X, grid, seeds):
    rows, best_per_m = [], []
    for m in grid:
        candidates = [fit_fcm(X, m, K, seed) for seed in seeds]
        fit = min(candidates, key=lambda r: r['objective'])
        xb = weighted_xie_beni(X, fit['U'], fit['centers'], m, fit['w'])
        fit['XB'] = xb
        best_per_m.append(fit)
        rows.append({'m': m, 'seed': fit['seed'], 'XB': xb, 'objective': fit['objective'],
                     'converged': fit['converged'], 'n_iter': fit['n_iter']})
    table = pd.DataFrame(rows)
    valid = table.XB.notna() & np.isfinite(table.XB)
    if not valid.any():
        raise RuntimeError('No finite FCM XB values.')
    idx = table.loc[valid, 'XB'].idxmin()
    return best_per_m[idx], table


def select_wfcm(X, grid, seeds, engine='stable', xb_min=None):
    gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=IS_SEED).fit(X)
    if not gmm.converged_:
        raise RuntimeError('Importance proposal GMM did not converge.')
    Z, _ = gmm.sample(NUM_IS)
    logq = gmm.score_samples(Z)
    # Independent validation draws from the same proposal, never used to tune.
    gmm.random_state = VALIDATION_SEED
    Zv, _ = gmm.sample(NUM_IS)
    logqv = gmm.score_samples(Zv)
    rows, fits = [], []
    for m in grid:
        candidates = []
        for seed in seeds:
            if engine == 'stable':
                f = fit_wfcm_stable(X, Z, logq, float(m), K, seed, MAX_OUTER, INNER_MAXITER)
            elif engine == 'legacy':
                f = fit_wfcm(X, Z, np.exp(logq), float(m), K, seed, MAX_OUTER)
                f.update(engine='legacy', seed=seed)
            else:
                raise ValueError('WFCM_ENGINE must be stable or legacy.')
            if np.isfinite(f['NLL']) and np.isfinite(f['U']).all():
                candidates.append(f)
        if not candidates:
            rows.append({'m': m, 'WXB': np.inf, 'eligible': False})
            fits.append(None)
            continue
        f = min(candidates, key=lambda r: r['NLL'])
        U_for_xb = f['U_raw'] if engine == 'legacy' else f['U']
        xb = weighted_xie_beni(X, U_for_xb, f['centers'], m, f['w'])
        f['WXB'] = xb
        eligible = bool(np.isfinite(xb) and (xb_min is None or xb >= xb_min))
        rows.append({'m': m, 'seed': f['seed'], 'WXB': xb, 'NLL': f['NLL'],
                     'eligible': eligible, 'converged': f['converged'],
                     'optimizer_failures': f['optimizer_failures']})
        fits.append(f)
        print(f'WFCM {engine}: m={m:.4g}, WXB={xb:.6g}, converged={f["converged"]}', flush=True)
    table = pd.DataFrame(rows)
    if not table.eligible.any():
        raise RuntimeError('No eligible WFCM fit: ' + table.to_string(index=False))
    selected = fits[table.loc[table.eligible, 'WXB'].idxmin()]
    if engine == 'stable':
        train = importance_diagnostics(selected['theta'], Z, logq, selected['m'], K, X.shape[1])
        validation = importance_diagnostics(selected['theta'], Zv, logqv, selected['m'], K, X.shape[1])
        selected['IS_diagnostics'] = pd.DataFrame([train, validation], index=['fit_draws', 'independent_draws'])
        selected['validation_logZ_delta'] = validation['logZ']-train['logZ']
    else:
        selected['IS_diagnostics'] = pd.DataFrame([{'legacy_pdf_floor_fraction': float(np.mean(logq < np.log(1e-8)))}])
    if not selected['converged'] or selected['optimizer_failures']:
        warnings.warn('WFCM optimizer/convergence problems: inspect history before interpreting results.')
    return selected, table



## 5. One evaluation function for all three methods
Entropy is always returned in BOTH nats and normalized units.
K-means memberships below are hard one-hot assignments, not probabilities.

In [ ]:
def evaluate_method(fit, labels, method):
    U = fit['U']
    if U.shape != (len(labels), K) or not np.isfinite(U).all() or np.any(U < 0) or not np.allclose(U.sum(axis=1), 1):
        raise ValueError('Invalid membership matrix.')
    y = labels.to_numpy()
    hard = U.argmax(axis=1)
    contingency = np.array([[np.sum((y == s) & (hard == k)) for k in range(K)] for s in SUBTYPES])
    a, b = linear_sum_assignment(-contingency)
    mapping = {int(j): SUBTYPES[i] for i, j in zip(a, b)}
    colorder = [next(k for k in range(K) if mapping[k] == s) for s in SUBTYPES]
    cm = pd.DataFrame(contingency[:, colorder], index=pd.Index(SUBTYPES, name='PAM50'),
                      columns=pd.Index(SUBTYPES, name=method))
    recall = pd.Series(np.diag(cm)/cm.sum(axis=1), index=cm.index, name=method)
    members = pd.DataFrame(U[:, colorder], index=labels.index, columns=SUBTYPES)
    members['PAM50'] = y
    members['cluster_id'] = hard
    members['mapped_cluster'] = [mapping[j] for j in hard]
    members['max_membership'] = U.max(axis=1)
    members['uncertainty'] = 1-U.max(axis=1)
    members['entropy_nats'] = -xlogy(U, U).sum(axis=1)
    members['entropy_normalized'] = members.entropy_nats/np.log(K)
    order = np.argsort(-members[SUBTYPES].to_numpy(), axis=1, kind='stable')
    members['top1_cluster'] = np.array(SUBTYPES)[order[:, 0]]
    members['top2_cluster'] = np.array(SUBTYPES)[order[:, 1]]
    sorted_u = np.sort(U, axis=1)
    members['top_gap'] = sorted_u[:, -1]-sorted_u[:, -2]
    members['LumA_LumB_gap'] = abs(members.LumA-members.LumB)
    boundary = members.loc[members.PAM50.isin(['LumA','LumB']) &
                           members.top1_cluster.isin(['LumA','LumB']) &
                           members.top2_cluster.isin(['LumA','LumB'])].sort_values('top_gap')
    sizes = pd.DataFrame({'mapped_subtype': [mapping[j] for j in range(K)],
                          'hard_n': np.bincount(hard, minlength=K),
                          'soft_mass': U.sum(axis=0),
                          'estimated_distance_weight': fit.get('w', np.full(K, np.nan))},
                         index=pd.Index(range(K), name='cluster_id'))
    if method != 'WFCM':
        sizes['estimated_distance_weight'] = np.nan
    if (sizes.hard_n == 0).any():
        warnings.warn(f'{method}: empty hard cluster; its subtype name is arbitrary.')
    summary = {'method': method, 'm': fit.get('m', np.nan),
               'ARI': adjusted_rand_score(y, hard), 'NMI': normalized_mutual_info_score(y, hard),
               'hard_clusters': int(np.count_nonzero(sizes.hard_n)),
               'mean_max_membership': members.max_membership.mean(),
               'mean_entropy_normalized': members.entropy_normalized.mean(),
               'converged': fit.get('converged', True)}
    summary.update({s+'_recall': recall[s] for s in SUBTYPES})
    return dict(summary=summary, mapping=mapping, confusion_matrix=cm,
                confusion_row_normalized=cm.div(cm.sum(axis=1), axis=0), subtype_recall=recall,
                membership_df=members, raw_memberships=pd.DataFrame(U, index=labels.index, columns=[f'C{k}' for k in range(K)]),
                soft_confusion=members.groupby('PAM50')[SUBTYPES].mean().reindex(SUBTYPES),
                uncertainty_summary=members.groupby('PAM50')[['max_membership','uncertainty','entropy_nats','entropy_normalized']].agg(['mean','median']).reindex(SUBTYPES),
                cluster_sizes=sizes, luminal_boundary=boundary)


def plot_comparison(evaluations, X, labels):
    methods = list(evaluations)
    fig, axes = plt.subplots(2, len(methods), figsize=(14, 8), constrained_layout=True)
    for j, method in enumerate(methods):
        for row, key, title in [(0, 'confusion_row_normalized', 'Hard recall by subtype'),
                                 (1, 'soft_confusion', 'Mean membership')]:
            values = evaluations[method][key].to_numpy()
            ax = axes[row, j]
            ax.imshow(values, cmap='Blues', vmin=0, vmax=1)
            ax.set(xticks=range(K), yticks=range(K), xticklabels=SUBTYPES, yticklabels=SUBTYPES,
                   title=method+' — '+title, xlabel='Mapped cluster', ylabel='PAM50')
            for a in range(K):
                for b in range(K):
                    ax.text(b,a,f'{values[a,b]:.3f}',ha='center',va='center',color='white' if values[a,b]>.5 else 'black')
    fig2, axes2 = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
    groups = [('PAM50', labels.to_numpy())] + [(m, evaluations[m]['membership_df'].mapped_cluster.to_numpy()) for m in methods]
    for ax, (name, group) in zip(axes2, groups):
        for s, color in zip(SUBTYPES, ['#0072B2','#D55E00','#009E73','#CC79A7']):
            mask = group == s
            ax.scatter(X[mask,0],X[mask,1],s=10,alpha=.6,label=s,color=color)
        ax.set(title=name, xlabel='PC1', ylabel='PC2')
    axes2[-1].legend(fontsize=8)
    return fig, fig2



## 6. Load or reuse the input
When using a NumPy X_pca in existing mode, its row order must already match
labels. Raw/prepared modes establish the alignment explicitly.

In [ ]:
selected_genes, pca_model = [], None
if INPUT_MODE == 'raw':
    X_pca, labels, analysis_info, pca_model, selected_genes = load_raw_data(
        EXPR_PATH, CLINICAL_SOURCE, STANDARDIZE_GENES)
elif INPUT_MODE == 'prepared':
    prepared = pd.read_csv(PREPARED_PATH, index_col=0)
    pc_cols = [f'PC{i+1}' for i in range(N_PC)]
    X_pca, labels = prepared[pc_cols], prepared.PAM50.copy()
elif INPUT_MODE == 'existing':
    if 'X_pca' not in globals() or 'labels' not in globals():
        raise ValueError('Existing mode needs X_pca and labels in this kernel.')
else:
    raise ValueError('INPUT_MODE must be raw, prepared, or existing.')
X_pca, labels = check_input(X_pca, labels)
input_hash = hashlib.sha256(X_pca.astype('<f8').tobytes() + '\n'.join(map(str, labels.index)).encode()).hexdigest()
print('Input:', X_pca.shape, 'SHA256:', input_hash)
print(labels.value_counts())
if pca_model is not None:
    print('First two PC variance fractions:', pca_model.explained_variance_ratio_[:2])



## 7. Fit all methods before inspecting external performance
Fixed m=1.1 is the default requested comparison. In selection mode both fuzzy
methods use M_GRID; FCM uses XB and WFCM uses WXB. Multiple restarts (if configured)
are chosen by distortion for FCM and NLL for WFCM, never by PAM50 agreement.

In [ ]:
if MODE not in ['fixed', 'select']:
    raise ValueError('MODE must be fixed or select.')
grid = [FIXED_M] if MODE == 'fixed' else list(M_GRID)
if not grid or np.any(np.asarray(grid) <= 1) or not np.isfinite(grid).all():
    raise ValueError('All m values must be finite and strictly greater than 1.')
kmeans = KMeans(n_clusters=K, n_init=100, random_state=123).fit(X_pca)
km_fit = {'U': np.eye(K)[kmeans.labels_], 'centers': kmeans.cluster_centers_,
          'converged': bool(kmeans.n_iter_ < kmeans.max_iter)}
fcm_fit, fcm_search = select_fcm(X_pca, grid, FCM_SEEDS)
# wfcm_fit, wfcm_search = select_wfcm(X_pca, grid, WFCM_SEEDS, WFCM_ENGINE, None if MODE == 'fixed' else WXB_MIN)


# Adaptive WXB floor: exclude m values where WXB < 5% of near-hard baseline.
# This prevents WXB from trivially selecting the highest m where u_ij^m → 0.
if MODE == 'select':
    _raw_fit, _raw_search = select_wfcm(
        X_pca, list(M_GRID), WFCM_SEEDS, WFCM_ENGINE, xb_min=None)
    _finite_wxb = _raw_search['WXB'].replace([np.inf], np.nan).dropna()
    _wxb_floor  = 0.05 * _finite_wxb.iloc[0]   # 5% of WXB at smallest m
    _eligible   = _raw_search['WXB'] >= _wxb_floor
    if not _eligible.any():
        raise RuntimeError('No m eligible after adaptive WXB floor.')
    # Use the correct row value
    _best_m = float(_raw_search.loc[_raw_search.loc[_eligible, 'WXB'].idxmin(), 'm'])
    print(f"WXB floor = {_wxb_floor:.5g};  selected m = {_best_m:.4g}")
    wfcm_fit, wfcm_search = select_wfcm(
        X_pca, [_best_m], WFCM_SEEDS, WFCM_ENGINE, xb_min=None)
else:
    wfcm_fit, wfcm_search = select_wfcm(
        X_pca, [FIXED_M], WFCM_SEEDS, WFCM_ENGINE, xb_min=None)





fits = {'K-means': km_fit, 'FCM': fcm_fit, 'WFCM': wfcm_fit}
display(fcm_search)
display(wfcm_search)
display(wfcm_fit['history'])
display(wfcm_fit['IS_diagnostics'])
if 'validation_logZ_delta' in wfcm_fit:
    print('Independent minus fit-draw logZ:', wfcm_fit['validation_logZ_delta'])
    print('Weight/sigma bound reached:', wfcm_fit['weight_bound_hit'], wfcm_fit['sigma_bound_hit'])
print('Inspect convergence, importance ESS, largest weight, and independent logZ agreement before interpreting WFCM.')



## 8. Compare results and save all patient-level outputs
Poor WFCM performance does not by itself imply a coding error, and corrected
arithmetic does not guarantee improved ARI/NMI or biologically meaningful softness.

In [ ]:
evaluations = {name: evaluate_method(fit, labels, name) for name, fit in fits.items()}
comparison = pd.DataFrame([evaluations[name]['summary'] for name in fits]).set_index('method')
display(comparison.round(4))
for name, result in evaluations.items():
    print('\n', name, 'mapping:', result['mapping'])
    for key in ['confusion_matrix','soft_confusion','cluster_sizes','uncertainty_summary']:
        print(key)
        display(result[key].round(4))
fig_heatmap, fig_pca = plot_comparison(evaluations, X_pca, labels)
plt.show()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(OUTPUT_DIR/'comparison.csv')
fcm_search.to_csv(OUTPUT_DIR/'fcm_search.csv', index=False)
wfcm_search.to_csv(OUTPUT_DIR/'wfcm_search.csv', index=False)
wfcm_fit['history'].to_csv(OUTPUT_DIR/'wfcm_optimization.csv', index=False)
wfcm_fit['IS_diagnostics'].to_csv(OUTPUT_DIR/'wfcm_importance_diagnostics.csv')
for name, result in evaluations.items():
    stem = name.lower().replace('-', '')
    for key, value in result.items():
        if isinstance(value, (pd.DataFrame, pd.Series)):
            value.to_csv(OUTPUT_DIR/f'{stem}_{key}.csv')
    np.savez_compressed(OUTPUT_DIR/f'{stem}_fit.npz', U=fits[name]['U'],
                        centers=fits[name]['centers'], w=fits[name].get('w', np.array([])),
                        m=fits[name].get('m', np.nan), sigma=fits[name].get('sigma', np.nan))
prepared = pd.DataFrame(X_pca, index=labels.index, columns=[f'PC{i+1}' for i in range(N_PC)])
prepared['PAM50'] = labels.to_numpy()
prepared.to_csv(OUTPUT_DIR/'prepared_pcs.csv')
fig_heatmap.savefig(OUTPUT_DIR/'comparison_heatmaps.png', dpi=180)
fig_pca.savefig(OUTPUT_DIR/'comparison_pca.png', dpi=180)
metadata = {'input_mode': INPUT_MODE, 'input_sha256': input_hash,
            'standardize_genes_if_raw': STANDARDIZE_GENES, 'selected_genes': selected_genes,
            'expression_path': str(EXPR_PATH), 'clinical_source': str(CLINICAL_SOURCE),
            'shape': list(X_pca.shape), 'mode': MODE, 'm_grid': list(map(float,grid)),
            'wfcm_engine': WFCM_ENGINE, 'wxb_min_if_selection': WXB_MIN,
            'fcm_seeds': FCM_SEEDS, 'wfcm_seeds': WFCM_SEEDS, 'is_samples': NUM_IS,
            'is_seed': IS_SEED, 'validation_seed': VALIDATION_SEED,
            'max_outer': MAX_OUTER, 'inner_maxiter': INNER_MAXITER,
            'python': platform.python_version(), 'numpy': np.__version__,
            'scipy': scipy.__version__, 'pandas': pd.__version__, 'sklearn': sklearn.__version__}
(OUTPUT_DIR/'run_settings.json').write_text(json.dumps(metadata, indent=2))
print('Saved comparison and patient-level results to', OUTPUT_DIR.resolve())


## Supplementary embedding and membership diagnostics

In [ ]:
# Optional embedding dependency: install umap-learn before running this section.
# The following import guard skips UMAP panels when it is unavailable.


In [ ]:
# ── Cell S1: Install / import extra dependencies ──────────────────────────────
import os, gc, warnings
 
# Force numba / OpenMP to single thread — the most reliable crash fix
os.environ.setdefault('NUMBA_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('NUMBA_DISABLE_JIT', '0')   # keep JIT but single-threaded
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

In [ ]:
# ── import UMAP with crash guard ──────────────────────────────────────────────
UMAP_AVAILABLE = False
try:
    import umap as umap_module          # import the module, not the class
    UMAP_AVAILABLE = True
    print('umap-learn imported OK.')
except Exception as e:
    warnings.warn(f'UMAP import failed ({e}). UMAP panels will be skipped; '
                  'all other supplementary figures will still be produced.')
 
SUBTYPE_COLORS = {
    'Basal': '#0072B2',
    'Her2':  '#D55E00',
    'LumA':  '#009E73',
    'LumB':  '#CC79A7',
}
 
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)   # already defined in main notebook
 

In [ ]:
# ── Cell S2: Compute alternative embeddings ───────────────────────────────────
print('Computing t-SNE…', flush=True)
tsne = TSNE(n_components=2, perplexity=40, learning_rate='auto',
            init='pca', random_state=42, max_iter=1000)
Z_tsne = tsne.fit_transform(X_pca)
print(f't-SNE KL divergence: {tsne.kl_divergence_:.4f}')
 
if UMAP_AVAILABLE:
    print('Computing UMAP…', flush=True)
    reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.3,
                        metric='euclidean', random_state=42)
    Z_umap = reducer.fit_transform(X_pca)
    print('UMAP done.')
else:
    Z_umap = None
 
embedding_dict = {'PCA (PC1–PC2)': X_pca[:, :2], 't-SNE': Z_tsne}
if Z_umap is not None:
    embedding_dict['UMAP'] = Z_umap
 
pd.DataFrame(Z_tsne, index=labels.index,
             columns=['tSNE1', 'tSNE2']).assign(PAM50=labels.to_numpy()).to_csv(
    OUTPUT_DIR / 'tsne_coords.csv')
if Z_umap is not None:
    pd.DataFrame(Z_umap, index=labels.index,
                 columns=['UMAP1', 'UMAP2']).assign(PAM50=labels.to_numpy()).to_csv(
        OUTPUT_DIR / 'umap_coords.csv')
 
 

In [ ]:
# ── Cell S3: Figure S1 — Problem difficulty: LumA/LumB overlap in all spaces ─
#
# Reviewer claim: "clusters are well-separated."
# Counter-evidence: LumA and LumB remain intermixed in every embedding, while
# Basal is the only truly separated subtype.  ARI ≈ 0.36 (well below 1.0)
# confirms the task is non-trivial.
 
n_emb = len(embedding_dict)
fig, axes = plt.subplots(2, n_emb, figsize=(5 * n_emb, 9),
                         constrained_layout=True)
fig.suptitle('Figure S1 — PAM50 subtypes in alternative embeddings\n'
             'LumA/LumB overlap persists across PCA, t-SNE, and UMAP, '
             'demonstrating that the problem is NOT trivially separable.',
             fontsize=11)
 
y_arr = labels.to_numpy()
 
for col, (name, Z) in enumerate(embedding_dict.items()):
    # Top row: all four subtypes
    ax_all = axes[0, col]
    for s in SUBTYPES:
        mask = y_arr == s
        ax_all.scatter(Z[mask, 0], Z[mask, 1], s=12, alpha=0.6,
                       color=SUBTYPE_COLORS[s], label=s, rasterized=True)
    ax_all.set(title=name, xlabel='Dim 1', ylabel='Dim 2')
    if col == n_emb - 1:
        ax_all.legend(fontsize=8, loc='upper right')
 
    # Bottom row: LumA vs LumB only, with pairwise distance density
    ax_lum = axes[1, col]
    for s, alpha in [('LumA', 0.5), ('LumB', 0.6)]:
        mask = y_arr == s
        ax_lum.scatter(Z[mask, 0], Z[mask, 1], s=14, alpha=alpha,
                       color=SUBTYPE_COLORS[s], label=s, rasterized=True)
    ax_lum.set(title=f'{name} — LumA vs LumB only',
               xlabel='Dim 1', ylabel='Dim 2')
    if col == n_emb - 1:
        ax_lum.legend(fontsize=8)
 
axes[0, 0].set_ylabel('All subtypes\n\nDim 2', fontsize=9)
axes[1, 0].set_ylabel('LumA / LumB\n\nDim 2', fontsize=9)
 
fig.savefig(OUTPUT_DIR / 'figS1_embedding_overlap.pdf', dpi=200)
fig.savefig(OUTPUT_DIR / 'figS1_embedding_overlap.png', dpi=200)
plt.show()
print('Saved Fig S1.')
 
 

In [ ]:
# ── Cell S4: Figure S2 — Cluster assignments overlaid on each embedding ───────
#
# Shows that K-means splits the luminal region with a hard boundary while
# WFCM and FCM assign graded memberships there.
 
methods_ordered = ['K-means', 'FCM', 'WFCM']
n_methods = len(methods_ordered)
 
fig, axes = plt.subplots(n_emb, n_methods + 1, figsize=(4.5 * (n_methods + 1), 4.5 * n_emb),
                         constrained_layout=True)
if axes.ndim == 1:
    axes = axes[np.newaxis, :]
 
fig.suptitle('Figure S2 — Hard cluster assignments in alternative embeddings\n'
             'Per-subtype recall is calculated from the current run.',
             fontsize=11)
 
for row, (emb_name, Z) in enumerate(embedding_dict.items()):
    # Column 0: PAM50 ground truth
    ax_gt = axes[row, 0]
    for s in SUBTYPES:
        mask = y_arr == s
        ax_gt.scatter(Z[mask, 0], Z[mask, 1], s=10, alpha=0.6,
                      color=SUBTYPE_COLORS[s], label=s, rasterized=True)
    ax_gt.set(title='PAM50 truth' if row == 0 else '',
              xlabel='Dim 1', ylabel=f'{emb_name}\nDim 2')
    if row == 0:
        ax_gt.legend(fontsize=7, loc='upper right')
 
    for col, method in enumerate(methods_ordered, start=1):
        ax = axes[row, col]
        mapped = evaluations[method]['membership_df']['mapped_cluster'].to_numpy()
        for s in SUBTYPES:
            mask = mapped == s
            ax.scatter(Z[mask, 0], Z[mask, 1], s=10, alpha=0.6,
                       color=SUBTYPE_COLORS[s], label=s, rasterized=True)
        recall_str = '  '.join(
            f'{s[0]}:{evaluations[method]["subtype_recall"][s]:.2f}'
            for s in SUBTYPES)
        ax.set(title=f'{method}\n{recall_str}' if row == 0 else '',
               xlabel='Dim 1')
 
fig.savefig(OUTPUT_DIR / 'figS2_cluster_assignments.pdf', dpi=200)
fig.savefig(OUTPUT_DIR / 'figS2_cluster_assignments.png', dpi=200)
plt.show()
print('Saved Fig S2.')

In [ ]:
# ── Cell S5: Figure S3 — Membership uncertainty in embedding space ────────────
#
# WFCM's core advantage: graded memberships at the LumA/LumB boundary
# identify patients that are biologically ambiguous, not mis-classified.
# K-means forces a hard choice for every patient; FCM spreads uncertainty
# more broadly but less selectively.
 
fig, axes = plt.subplots(n_emb, n_methods, figsize=(4.5 * n_methods, 4.5 * n_emb),
                         constrained_layout=True)
if axes.ndim == 1:
    axes = axes[np.newaxis, :]
 
fig.suptitle('Figure S3 — Membership uncertainty (1 – max membership) in each embedding\n'
             'WFCM concentrates uncertainty at the genuine LumA/LumB interface; '
             'K-means cannot express uncertainty.',
             fontsize=11)
 
uncertainty_cmap = plt.cm.plasma
 
for row, (emb_name, Z) in enumerate(embedding_dict.items()):
    for col, method in enumerate(methods_ordered):
        ax = axes[row, col]
        unc = evaluations[method]['membership_df']['uncertainty'].to_numpy()
        sc = ax.scatter(Z[:, 0], Z[:, 1], c=unc, cmap=uncertainty_cmap,
                        s=10, alpha=0.8, vmin=0, vmax=1, rasterized=True)
        ax.set(title=f'{method}' if row == 0 else '',
               xlabel='Dim 1',
               ylabel=f'{emb_name}\nDim 2' if col == 0 else '')
        if col == n_methods - 1:
            cbar = plt.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
            cbar.set_label('Uncertainty\n(1 – max membership)', fontsize=8)
 
fig.savefig(OUTPUT_DIR / 'figS3_uncertainty_embedding.pdf', dpi=200)
fig.savefig(OUTPUT_DIR / 'figS3_uncertainty_embedding.png', dpi=200)
plt.show()
print('Saved Fig S3.')
 

In [ ]:
# ── Cell S6: Figure S4 — LumB-specific membership (WFCM advantage) ───────────
#
# WFCM's LumB recall (0.88) far exceeds K-means (0.56) and FCM (0.45).
# This panel shows LumB membership continuously, revealing that WFCM
# "finds" LumB patients that hard clustering scatters into LumA.
 
fig, axes = plt.subplots(n_emb, n_methods, figsize=(4.5 * n_methods, 4.5 * n_emb),
                         constrained_layout=True)
if axes.ndim == 1:
    axes = axes[np.newaxis, :]
 
fig.suptitle('Figure S4 — LumB soft membership across embeddings\n'
             'WFCM assigns high LumB membership to true LumB patients '
             'that K-means and FCM misclassify as LumA.',
             fontsize=11)
 
lumB_cmap = plt.cm.YlOrRd
 
for row, (emb_name, Z) in enumerate(embedding_dict.items()):
    for col, method in enumerate(methods_ordered):
        ax = axes[row, col]
        mdf = evaluations[method]['membership_df']
        lumB_mem = mdf['LumB'].to_numpy()
        sc = ax.scatter(Z[:, 0], Z[:, 1], c=lumB_mem, cmap=lumB_cmap,
                        s=10, alpha=0.8, vmin=0, vmax=1, rasterized=True)
        recall_val = evaluations[method]['subtype_recall']['LumB']
        ax.set(title=f'{method}  (LumB recall = {recall_val:.3f})' if row == 0 else '',
               xlabel='Dim 1',
               ylabel=f'{emb_name}\nDim 2' if col == 0 else '')
        # Outline true LumB patients
        true_lumB = y_arr == 'LumB'
        ax.scatter(Z[true_lumB, 0], Z[true_lumB, 1], s=22, facecolors='none',
                   edgecolors='black', linewidths=0.4, alpha=0.6, rasterized=True,
                   label='True LumB' if (row == 0 and col == 0) else '')
        if col == n_methods - 1:
            cbar = plt.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
            cbar.set_label('LumB membership', fontsize=8)
 
axes[0, 0].legend(fontsize=7)
fig.savefig(OUTPUT_DIR / 'figS4_lumB_membership.pdf', dpi=200)
fig.savefig(OUTPUT_DIR / 'figS4_lumB_membership.png', dpi=200)
plt.show()
print('Saved Fig S4.')
 

In [ ]:
# ── Cell S7: Figure S5 — Subtype-level recall comparison (bar chart) ─────────
 
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
fig.suptitle('Figure S5 — Per-subtype recall and global metrics across methods',
             fontsize=11)
 
# Left: subtype recall bars
ax = axes[0]
bar_width = 0.22
x = np.arange(len(SUBTYPES))
for i, method in enumerate(methods_ordered):
    recalls = [evaluations[method]['subtype_recall'][s] for s in SUBTYPES]
    bars = ax.bar(x + i * bar_width, recalls, bar_width, label=method,
                  color=['#4E79A7', '#F28E2B', '#E15759'][i], alpha=0.85)
ax.axhline(0.8, color='grey', linewidth=0.8, linestyle='--', label='0.80 reference')
ax.set(xticks=x + bar_width, xticklabels=SUBTYPES, ylabel='Hard recall',
       title='Per-subtype recall', ylim=(0, 1.05))
ax.legend(fontsize=9)
for tick_label, s in zip(ax.get_xticklabels(), SUBTYPES):
    tick_label.set_color(SUBTYPE_COLORS[s])
 
# Annotate WFCM LumB bar
wfcm_lumB_val = evaluations['WFCM']['subtype_recall']['LumB']
lumB_idx = SUBTYPES.index('LumB')
ax.annotate(f'+{wfcm_lumB_val - evaluations["K-means"]["subtype_recall"]["LumB"]:.2f} vs K-means',
            xy=(lumB_idx + 2 * bar_width, wfcm_lumB_val),
            xytext=(lumB_idx + 2 * bar_width - 0.5, wfcm_lumB_val + 0.07),
            fontsize=7.5, color='#E15759',
            arrowprops=dict(arrowstyle='->', color='#E15759', lw=1))
 
# Right: global ARI / NMI
ax2 = axes[1]
global_metrics = {
    'ARI':  {m: evaluations[m]['summary']['ARI'] for m in methods_ordered},
    'NMI':  {m: evaluations[m]['summary']['NMI'] for m in methods_ordered},
}
metric_x = np.arange(len(global_metrics))
for i, method in enumerate(methods_ordered):
    vals = [global_metrics[mk][method] for mk in global_metrics]
    ax2.bar(metric_x + i * bar_width, vals, bar_width, label=method,
            color=['#4E79A7', '#F28E2B', '#E15759'][i], alpha=0.85)
ax2.set(xticks=metric_x + bar_width, xticklabels=list(global_metrics.keys()),
        ylabel='Score', title='Global clustering metrics', ylim=(0, 0.65))
ax2.legend(fontsize=9)
 
fig.savefig(OUTPUT_DIR / 'figS5_recall_metrics.pdf', dpi=200)
fig.savefig(OUTPUT_DIR / 'figS5_recall_metrics.png', dpi=200)
plt.show()
print('Saved Fig S5.')
 
 